# SpaceX Falcon 9 - First Stage Landing Prediction (Classification)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

In [ ]:
def plot_confusion_matrix(y, y_predict, title='Confusion Matrix'):
    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot()
    sns.heatmap(cm, annot=True, ax=ax, fmt='g')
    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.set_title(title)
    ax.xaxis.set_ticklabels(['did not land', 'landed'])
    ax.yaxis.set_ticklabels(['did not land', 'landed'])
    plt.show()

In [ ]:
data = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv')
X = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv')
data.head()

## Task 1: Create the target array

In [ ]:
Y = data['Class'].to_numpy()
Y[:10]

## Task 2: Standardize the features

In [ ]:
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)
X[0:3]

## Task 3: Train/test split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print('Test samples:', Y_test.shape[0])

## Task 4: Logistic Regression

In [ ]:
parameters = {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}
logreg_cv = GridSearchCV(LogisticRegression(), parameters, cv=10)
logreg_cv.fit(X_train, Y_train)
print('Best params:', logreg_cv.best_params_)
print('Validation accuracy:', logreg_cv.best_score_)

## Task 5: Test accuracy and confusion matrix (Logistic Regression)

In [ ]:
acc_lr = logreg_cv.score(X_test, Y_test)
print('Test accuracy:', acc_lr)
plot_confusion_matrix(Y_test, logreg_cv.predict(X_test), 'Logistic Regression')

## Task 6: Support Vector Machine

In [ ]:
parameters = {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'),
              'C': np.logspace(-3, 3, 5),
              'gamma': np.logspace(-3, 3, 5)}
svm_cv = GridSearchCV(SVC(), parameters, cv=10)
svm_cv.fit(X_train, Y_train)
print('Best params:', svm_cv.best_params_)
print('Validation accuracy:', svm_cv.best_score_)

In [ ]:
acc_svm = svm_cv.score(X_test, Y_test)
print('Test accuracy:', acc_svm)
plot_confusion_matrix(Y_test, svm_cv.predict(X_test), 'SVM')

## Task 8: Decision Tree

In [ ]:
parameters = {'criterion': ['gini', 'entropy'],
              'splitter': ['best', 'random'],
              'max_depth': [2 * n for n in range(1, 10)],
              'max_features': ['sqrt', 'log2'],
              'min_samples_leaf': [1, 2, 4],
              'min_samples_split': [2, 5, 10]}
tree_cv = GridSearchCV(DecisionTreeClassifier(random_state=1), parameters, cv=10)
tree_cv.fit(X_train, Y_train)
print('Best params:', tree_cv.best_params_)
print('Validation accuracy:', tree_cv.best_score_)

In [ ]:
acc_tree = tree_cv.score(X_test, Y_test)
print('Test accuracy:', acc_tree)
plot_confusion_matrix(Y_test, tree_cv.predict(X_test), 'Decision Tree')

## Task 10: K Nearest Neighbors

In [ ]:
parameters = {'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
              'p': [1, 2]}
knn_cv = GridSearchCV(KNeighborsClassifier(), parameters, cv=10)
knn_cv.fit(X_train, Y_train)
print('Best params:', knn_cv.best_params_)
print('Validation accuracy:', knn_cv.best_score_)

In [ ]:
acc_knn = knn_cv.score(X_test, Y_test)
print('Test accuracy:', acc_knn)
plot_confusion_matrix(Y_test, knn_cv.predict(X_test), 'KNN')

## Task 12: Compare all models

In [ ]:
results = pd.DataFrame({'Model': ['Logistic Regression', 'SVM', 'Decision Tree', 'KNN'],
                        'Test Accuracy': [acc_lr, acc_svm, acc_tree, acc_knn],
                        'Validation Accuracy': [logreg_cv.best_score_, svm_cv.best_score_, tree_cv.best_score_, knn_cv.best_score_]})
print(results)
best = results.sort_values('Test Accuracy', ascending=False).iloc[0]
print('Best model on test data:', best['Model'], best['Test Accuracy'])

results.set_index('Model').plot(kind='bar', figsize=(9, 5))
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.show()